# 📦 Notebook 3: Denormalization

When indexes aren't enough, denormalization trades storage for speed by eliminating expensive joins.

## Learning Objectives

By the end of this notebook, you'll understand:
- Normalized vs denormalized schemas
- When to denormalize
- Materialized views
- Pre-computed aggregations

---

🔍 **Open Adminer** at http://localhost:8080 to see table structures and run queries!

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [1]:
import psycopg2
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def run_query(query: str):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    try:
        results = cursor.fetchall()
    except:
        results = []
    conn.commit()
    conn.close()
    return results

def measure_query(query: str) -> tuple:
    conn = get_connection()
    cursor = conn.cursor()
    start = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()
    return elapsed, results

print("✅ Connected to PostgreSQL")

✅ Connected to PostgreSQL


## 📊 Normalized vs Denormalized

**Normalization** eliminates data redundancy by splitting data across tables. **Denormalization** adds redundancy back for faster reads.

In [2]:
print("📊 Normalized vs Denormalized Schema")
print("=" * 60)
print("""
NORMALIZED (Our current schema)
─────────────────────────────────────────────────────────────
┌──────────┐     ┌──────────┐     ┌──────────┐
│  users   │     │  posts   │     │ comments │
├──────────┤     ├──────────┤     ├──────────┤
│ id       │◄────│ user_id  │◄────│ post_id  │
│ username │     │ content  │     │ user_id  │
│ email    │     │ likes    │     │ content  │
└──────────┘     └──────────┘     └──────────┘

To get a post with author info:
SELECT p.*, u.username FROM posts p JOIN users u ON p.user_id = u.id

─────────────────────────────────────────────────────────────

DENORMALIZED (Redundant but fast)
─────────────────────────────────────────────────────────────
┌─────────────────────────────┐
│       feed_items            │
├─────────────────────────────┤
│ post_id                     │
│ post_content                │
│ author_username  ◄─ COPY!   │
│ author_avatar    ◄─ COPY!   │
│ like_count                  │
└─────────────────────────────┘

To get a post with author info:
SELECT * FROM feed_items WHERE post_id = 123

─────────────────────────────────────────────────────────────
""")

print("💡 Denormalization eliminates JOINs by storing redundant data!")

📊 Normalized vs Denormalized Schema

NORMALIZED (Our current schema)
─────────────────────────────────────────────────────────────
┌──────────┐     ┌──────────┐     ┌──────────┐
│  users   │     │  posts   │     │ comments │
├──────────┤     ├──────────┤     ├──────────┤
│ id       │◄────│ user_id  │◄────│ post_id  │
│ username │     │ content  │     │ user_id  │
│ email    │     │ likes    │     │ content  │
└──────────┘     └──────────┘     └──────────┘

To get a post with author info:
SELECT p.*, u.username FROM posts p JOIN users u ON p.user_id = u.id

─────────────────────────────────────────────────────────────

DENORMALIZED (Redundant but fast)
─────────────────────────────────────────────────────────────
┌─────────────────────────────┐
│       feed_items            │
├─────────────────────────────┤
│ post_id                     │
│ post_content                │
│ author_username  ◄─ COPY!   │
│ author_avatar    ◄─ COPY!   │
│ like_count                  │
└─────────────────────

## 🔬 The Cost of Joins

In [3]:
print("🔬 Normalized Query: Feed with joins")
print("=" * 60)

normalized_query = """
SELECT 
    p.id, p.content, p.like_count, p.created_at,
    u.username, u.display_name, u.profile_image_url
FROM posts p
JOIN users u ON p.user_id = u.id
ORDER BY p.created_at DESC
LIMIT 20
"""

elapsed, results = measure_query(normalized_query)
print(f"\nTime: {elapsed:.2f}ms | Rows: {len(results)}")

print("\nSample result:")
if results:
    row = results[0]
    print(f"  Post ID: {row[0]}")
    print(f"  Content: {row[1][:50]}...")
    print(f"  Likes: {row[2]}")
    print(f"  Author: {row[4]}")

🔬 Normalized Query: Feed with joins

Time: 2.65ms | Rows: 20

Sample result:
  Post ID: 16839
  Content: Post content 16839. This is a sample post with som...
  Likes: 296
  Author: user7532


In [4]:
print("\n🔨 Creating denormalized feed_items table...")

run_query("DELETE FROM feed_items")

run_query("""
INSERT INTO feed_items (
    viewer_user_id, post_id, author_user_id,
    author_username, author_display_name, author_profile_image,
    post_content, post_image_url, like_count, comment_count, created_at
)
SELECT 
    1,  -- For demo, all items are for user 1
    p.id, p.user_id,
    u.username, u.display_name, u.profile_image_url,
    p.content, p.image_url, p.like_count, p.comment_count, p.created_at
FROM posts p
JOIN users u ON p.user_id = u.id
""")

run_query("CREATE INDEX IF NOT EXISTS idx_feed_viewer ON feed_items(viewer_user_id, created_at DESC)")

print("✅ Feed items created!")


🔨 Creating denormalized feed_items table...


✅ Feed items created!


In [5]:
print("🔬 Denormalized Query: Feed without joins")
print("=" * 60)

denormalized_query = """
SELECT 
    post_id, post_content, like_count, created_at,
    author_username, author_display_name, author_profile_image
FROM feed_items
WHERE viewer_user_id = 1
ORDER BY created_at DESC
LIMIT 20
"""

elapsed, results = measure_query(denormalized_query)
print(f"\nTime: {elapsed:.2f}ms | Rows: {len(results)}")

print("\n💡 No JOINs needed - all data is in one table!")

🔬 Denormalized Query: Feed without joins



Time: 0.96ms | Rows: 20

💡 No JOINs needed - all data is in one table!


## ⚖️ Trade-offs

In [6]:
print("⚖️ Denormalization Trade-offs")
print("=" * 60)
print("""
PROS:
─────────────────────────────────────────────────────────────
✅ Faster reads (no JOINs)
✅ Simpler queries
✅ Predictable performance
✅ Easier to cache

CONS:
─────────────────────────────────────────────────────────────
❌ More storage used
❌ Writes become complex (update multiple places)
❌ Data can become inconsistent
❌ Schema changes are harder

WHEN TO DENORMALIZE:
─────────────────────────────────────────────────────────────
• Read/write ratio > 100:1
• JOINs are killing performance
• Source data changes infrequently
• You can tolerate brief staleness

EXAMPLES:
─────────────────────────────────────────────────────────────
• Social media feeds (denormalize author info)
• E-commerce orders (denormalize product names)
• Analytics dashboards (denormalize everything!)
""")

⚖️ Denormalization Trade-offs

PROS:
─────────────────────────────────────────────────────────────
✅ Faster reads (no JOINs)
✅ Simpler queries
✅ Predictable performance
✅ Easier to cache

CONS:
─────────────────────────────────────────────────────────────
❌ More storage used
❌ Writes become complex (update multiple places)
❌ Data can become inconsistent
❌ Schema changes are harder

WHEN TO DENORMALIZE:
─────────────────────────────────────────────────────────────
• Read/write ratio > 100:1
• JOINs are killing performance
• Source data changes infrequently
• You can tolerate brief staleness

EXAMPLES:
─────────────────────────────────────────────────────────────
• Social media feeds (denormalize author info)
• E-commerce orders (denormalize product names)
• Analytics dashboards (denormalize everything!)



## 📊 Materialized Views

Materialized views are like cached query results that PostgreSQL manages for you.

In [7]:
print("📊 Creating Materialized View for Product Ratings")
print("=" * 60)

run_query("DROP MATERIALIZED VIEW IF EXISTS product_avg_ratings")

run_query("""
CREATE MATERIALIZED VIEW product_avg_ratings AS
SELECT 
    p.id,
    p.name,
    p.category,
    COALESCE(AVG(r.rating), 0) as avg_rating,
    COUNT(r.id) as review_count
FROM products p
LEFT JOIN reviews r ON p.id = r.product_id
GROUP BY p.id, p.name, p.category
""")

run_query("CREATE INDEX idx_product_ratings_category ON product_avg_ratings(category)")

print("✅ Materialized view created!")

📊 Creating Materialized View for Product Ratings
✅ Materialized view created!


In [8]:
print("\n🔬 Query Comparison: Regular vs Materialized View")
print("=" * 60)

regular_query = """
SELECT 
    p.id, p.name, AVG(r.rating) as avg_rating, COUNT(r.id)
FROM products p
LEFT JOIN reviews r ON p.id = r.product_id
WHERE p.category = 'Electronics'
GROUP BY p.id, p.name
ORDER BY avg_rating DESC
LIMIT 10
"""

materialized_query = """
SELECT id, name, avg_rating, review_count
FROM product_avg_ratings
WHERE category = 'Electronics'
ORDER BY avg_rating DESC
LIMIT 10
"""

elapsed1, results1 = measure_query(regular_query)
elapsed2, results2 = measure_query(materialized_query)

print(f"\nRegular query (JOIN + GROUP BY): {elapsed1:.2f}ms")
print(f"Materialized view query:          {elapsed2:.2f}ms")
print(f"\nSpeedup: {elapsed1/elapsed2:.1f}x faster!")


🔬 Query Comparison: Regular vs Materialized View

Regular query (JOIN + GROUP BY): 3.12ms
Materialized view query:          1.08ms

Speedup: 2.9x faster!


In [9]:
print("\n🔄 Refreshing Materialized Views")
print("=" * 60)
print("""
Materialized views are SNAPSHOTS - they don't auto-update!

REFRESH OPTIONS:
─────────────────────────────────────────────────────────────
1. Manual refresh (blocks reads during refresh):
   REFRESH MATERIALIZED VIEW product_avg_ratings;

2. Concurrent refresh (allows reads during refresh):
   REFRESH MATERIALIZED VIEW CONCURRENTLY product_avg_ratings;
   (Requires unique index on view)

3. Scheduled refresh via cron or background job:
   - Every 5 minutes for frequently changing data
   - Every hour for slowly changing data
   - Nightly for analytics views
""")

print("\n🔄 Refreshing our view...")
run_query("REFRESH MATERIALIZED VIEW product_avg_ratings")
print("✅ View refreshed!")


🔄 Refreshing Materialized Views

Materialized views are SNAPSHOTS - they don't auto-update!

REFRESH OPTIONS:
─────────────────────────────────────────────────────────────
1. Manual refresh (blocks reads during refresh):
   REFRESH MATERIALIZED VIEW product_avg_ratings;

2. Concurrent refresh (allows reads during refresh):
   REFRESH MATERIALIZED VIEW CONCURRENTLY product_avg_ratings;
   (Requires unique index on view)

3. Scheduled refresh via cron or background job:
   - Every 5 minutes for frequently changing data
   - Every hour for slowly changing data
   - Nightly for analytics views


🔄 Refreshing our view...
✅ View refreshed!


## 🧪 Quick Quiz

1. **When should you denormalize?**

2. **What's the downside of denormalization?**

3. **When do materialized views become stale?**

In [10]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. When to denormalize:")
print("   - High read/write ratio (>100:1)")
print("   - JOINs are bottleneck")
print("   - Data changes infrequently")
print("   - Can tolerate brief staleness")
print()
print("2. Downside of denormalization:")
print("   - Data duplication (storage cost)")
print("   - Write complexity (update multiple places)")
print("   - Potential inconsistency")
print("   - Harder schema changes")
print()
print("3. Materialized views become stale:")
print("   - Immediately after source data changes")
print("   - They're snapshots, not live queries")
print("   - Must explicitly REFRESH")

📝 Quiz Answers

1. When to denormalize:
   - High read/write ratio (>100:1)
   - JOINs are bottleneck
   - Data changes infrequently
   - Can tolerate brief staleness

2. Downside of denormalization:
   - Data duplication (storage cost)
   - Write complexity (update multiple places)
   - Potential inconsistency
   - Harder schema changes

3. Materialized views become stale:
   - Immediately after source data changes
   - They're snapshots, not live queries
   - Must explicitly REFRESH


## 📚 Summary

### Key Takeaways

1. **Denormalization trades writes for reads** - duplicate data for faster queries
2. **Use for high read/write ratios** - when reads dominate
3. **Materialized views** - database-managed cached aggregations
4. **Refresh strategy matters** - schedule based on staleness tolerance
5. **Not a silver bullet** - adds complexity to writes

### Next Up

In **Notebook 4**, we'll learn about read replicas:
- Leader-follower replication
- Handling replication lag
- Scaling beyond single server